# Shot size detection

<font color='red'> Problem with the following code: It only detects faces from the front. </font>

In [ ]:
import cv2
def detect_face(frame):
    """
    Detects a face (only from the front) in the given frame and returns its bounding box.
    Parameters:
        frame (numpy.ndarray): The current video frame.
    Returns:
        tuple: Bounding box of the face (x, y, w, h) if a face is detected, None otherwise.
    """
    # Load the pre-trained Haar Cascade face detector
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

    # Convert the frame to grayscale (Haar cascade works with grayscale images)
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Detect faces
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))

    # If at least one face is detected, return the first one
    if len(faces) > 0:
        return faces[0]  # (x, y, w, h)

    # No face detected
    return None

In [ ]:
def classify_shot(frame):
    """
    Classifies the shot type based on the face's size relative to the frame.
    Parameters:
        frame (numpy.ndarray): The current video frame.
    Returns:
        str: Shot type ("close-up", "medium shot", "long shot", or "no face detected").
    """
    face = detect_face(frame)
    
    if face is not None:
        face_height = face[3]  
        frame_height = frame.shape[1]  
        ratio = face_height / frame_height

        if 0.25 <= ratio:
            return "close-up"
        elif 0.1 < ratio < 0.25:
            return "medium shot"
        else:
            return "long shot"
    return "no face detected"

In [ ]:
def detect_shot_size (video_path):
    """
    Classifies the shot size in a video file.
    Parameters:
        video_path (file path): The path to a video file.
    Returns:
        For the time, only prints the shot sizes and returns nothing. 
    """
    cap = cv2.VideoCapture(video_path)
    # Initialize a counter for frames
    frame_index = 0  
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        
        # Process every tenth frame
        if frame_index % 10 == 0:
            shot_type = classify_shot(frame)
            print(shot_type)
        
        frame_index += 1
    
    cap.release()

# Jump cut detection

<font color='red'> Still very skeptical whether or not this code works properly. </font>

In [ ]:
import cv2, numpy as np  
def detect_jump_cuts(video_path, threshold=(0.5, 0.7)):  
    
    #cap is a VideoCapture object that allows frame-by-frame reading:
    cap = cv2.VideoCapture(video_path)
    
    """
    cap.read(): Reads the first frame from the video.
    ret: A boolean indicating whether reading was successful.
    prev: The first video frame as an image (NumPy array).
    """
    ret, prev = cap.read()  
    
    #Converts the first frame to grayscale using cv2.cvtColor(), which simplifies the histogram:
    prev_gray = cv2.cvtColor(prev, cv2.COLOR_BGR2GRAY)
    
    jump_cut_frames = []  
    while cap.isOpened():  
        #Reads the next frame from the video:
        ret, frame = cap.read() 
        
        #ret=False means the video is finished, so the loop breaks: 
        if not ret: break
            
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY) 

        #Calculates the correlation between the histograms of two grayscale frames: the current one and the previous one:
        diff = cv2.compareHist(cv2.calcHist([prev_gray], [0], None, [256], [0, 256]),  
                               cv2.calcHist([gray], [0], None, [256], [0, 256]), cv2.HISTCMP_CORREL) 
        
        #cap.get retrieves the current frame number. So number added to data is the number of the frame immediately after the cut. 
        if threshold[0] < diff < threshold[1]: jump_cut_frames.append(cap.get(cv2.CAP_PROP_POS_FRAMES))  
        prev_gray = gray  
    cap.release()  
    return jump_cut_frames 

### Use case:

In [ ]:
import os
directory = 'data/jump_cut'
file_names=[]

#create a list of single file names:
for filename in os.listdir(directory):
    f = os.path.join(directory, filename)
    # checking if it is a file
    if os.path.isfile(f):
        file_names.append(f)
file_names= sorted(file_names)

In [ ]:
file_path= file_names[3]
frame_list= detect_jump_cuts (file_path, threshold= (0.7, 0.9))
print (f"Jump cuts in '{file_path}' appear prior to the following frames: {frame_list}.")